# Các hàm funcion

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# Hàm load data
def load_data(file_path):
        data = pd.read_csv(file_path)
        return data

In [3]:
# Hàm chuẩn bị dữ liệu từ các thuộc tính mà người ta chọn. 
def Chuan_bi_du_lieu(list_feature, target, data_address):
    data = load_data(data_address)
    X= data[list_feature]
    y=data[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train,y_train,X_test,y_test

In [4]:
'''
    Đây là đoạn hàm lấy cái việc người dùng chọn new model hay new version.
        Nếu chọn new model thì gán list_model_search = [0,1,2,3] tức là lấy toàn bộ danh sách các model.
        Nếu chọn new version thì gán list_model_search là danh sách có 1 phần tử là id của mô hình người ta chọn. 
'''
def choose_model_version(choose):
    if(choose == "new model") : 
        list_model_search = [0,1,2,3]
    else:
        # Gọi đến hàm để người ta chọn xem người ta muốn train lại cái model nào. hàm này sẽ return ra id của mô hình đó. 
        # id = ...
        # list_model_search = [id]
        list_model_search = [2]
    return list_model_search

In [5]:
'''
    Hàm training:
        Đầu vào: 
            danh sách tất cả các mô hình - danh sách 1,
            danh sách các mô hình sẽ được training (nếu chọn new version thì danh sách này chỉ có 1 mô hình) - danh sách 2
            và matrix đã chọn. 
        Ta train những mô hình được chọn trong danh sách 2. 
        Ta lưu lại kết quả của từng mô hình. 
        Ta so sánh kết quả đó với nhau để chọn ra cái nào ngon nhất. 
        Đầu ra:
            id mô hình
            chính cái mô hình tốt nhất
            Score tốt nhất trên tập train
            params của mô hình đó. 

'''
def Training(models, list_model_search, matrix):
    best_model_id = None
    best_model = None
    best_score = -1
    best_params = {}
    
    for model_id in list_model_search:
        model, param_grid = models[model_id]
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring=matrix)
        grid_search.fit(X_train, y_train)
        
        if grid_search.best_score_ > best_score:
            best_model_id = model_id
            best_model = grid_search.best_estimator_
            best_score = grid_search.best_score_
            best_params = grid_search.best_params_

    return best_model_id, best_model ,best_score, best_params

In [6]:
'''
    Đây là hàm lấy cái việc người dùng chọn custom hay auto.
    Nếu chọn custom sẽ gọi đến hàm custom bên dưới
'''

''' 
    Đây là hàm hiển thị ra custom. 
'''

' \n    Đây là hàm hiển thị ra custom. \n'

# Minh họa Flow:  Chọn New Model hoặc new version => Auto => chọn thuộc tính... => Train
## Kiểu kiểu đây là hàm Main

In [7]:
data_address = "D:\Hoc\LAB\Example Auto ML\AUTOML week 1\glass.csv"


#Chọn new model.
choose = "new model"
list_model_search = choose_model_version(choose)

#Chọn auto

'''
    Hàm lấy ra tên của các thuộc tính , tên matrix và giờ training chọn ở phần fontend.
    Được kết quả như dưới này: 
'''
list_feature = ["RI","Na","Mg","Al","Si","K","Ca","Ba","Fe"]
target = "Type"
matrix = "accuracy"

# Gọi hàm chuẩn bị dữ liệu 
X_train,y_train,X_test,y_test = Chuan_bi_du_lieu(list_feature, target, data_address)

# Tạo danh sách các mô hình và tham số tìm kiếm của chúng. 
models = {
        0: (DecisionTreeClassifier(), {
            'max_depth': [None, 5, 10, 15],
            'min_samples_split': [2, 5, 10]
        }),
        1: (RandomForestClassifier(), {
            'n_estimators': [50, 100, 200],
            'max_features': ['auto', 'sqrt', 'log2']
        }),
        2: (KNeighborsClassifier(), {
            'n_neighbors': [3, 5, 7, 9],
            'weights': ['uniform', 'distance']
        }),
        3: (SVC(), {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf']
        })
    }

# Gọi hàm train
best_model_name, best_model ,best_score, best_params = Training(models,list_model_search, matrix)

print(f"Best Model ID: {best_model_name}")
print(f"Best Score: {best_score:.4f}")
print(f"Best Parameters: {best_params}")

best_model.fit(X_train, y_train)

test_score = best_model.score(X_test, y_test)
print(f"Test Score of Best Model: {test_score:.4f}")

d:\Software\anaconda\envs\myenv1\lib\site-packages\sklearn\model_selection\_validation.py:547: FitFailedWarning: 
15 fits failed out of a total of 45.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Software\anaconda\envs\myenv1\lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Software\anaconda\envs\myenv1\lib\site-packages\sklearn\base.py", line 1467, in wrapper
    estimator._validate_params()
  File "d:\Software\anaconda\envs\myenv1\lib\site-packages\sklearn\base.py", line 666, in _validate_params
    validate_parameter_constraints(
  File "d:\Software

Best Model ID: 1
Best Score: 0.7489
Best Parameters: {'max_features': 'log2', 'n_estimators': 200}
Test Score of Best Model: 0.8605
